# Text Feature Engineering — TF-IDF + LSA for Complaint-Topic Data

## Objective
Turn raw, unlabeled complaint/support-ticket text into a numeric feature matrix that any downstream algorithm (clustering, classification, search) can consume. This is a distinct skill from what happens *after* the numbers exist — the clustering method itself is covered separately in `kmeans_algorithms.ipynb`, which loads the feature matrix this notebook produces rather than rebuilding it.

**Proxy dataset:** the [20 Newsgroups](https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset) corpus — public, built into scikit-learn, real user-written posts — restricted to 7 product/service-oriented categories and reframed as complaint topics, a stand-in for triaging real incoming support-ticket text without any pre-existing labels:

| Newsgroup | Reframed as |
|---|---|
| `comp.sys.ibm.pc.hardware` | PC Hardware Complaints |
| `comp.sys.mac.hardware` | Mac Hardware Complaints |
| `rec.autos` | Automotive Complaints |
| `rec.motorcycles` | Motorcycle / Vehicle Complaints |
| `sci.med` | Health / Medical Product Complaints |
| `sci.electronics` | Electronics Product Complaints |
| `misc.forsale` | Billing / Marketplace Transaction Issues |

The newsgroup label is carried through to the saved output *only* so the downstream clustering notebook can validate its results against it — it plays no role in building the features below.

**Pipeline:** raw text -> TF-IDF (weighted term frequencies) -> Truncated SVD / LSA (dense, reduced-dimension representation) -> saved to `data/` for reuse.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

DATA_DIR = "."  # this notebook lives inside data/ itself — see the project README for why
RANDOM_STATE = 42

## 1. Load the Raw Text

In [2]:
CATEGORY_LABELS = {
    "comp.sys.ibm.pc.hardware": "PC Hardware",
    "comp.sys.mac.hardware":    "Mac Hardware",
    "rec.autos":                "Automotive",
    "rec.motorcycles":          "Motorcycle/Vehicle",
    "sci.med":                  "Health/Medical Product",
    "sci.electronics":          "Electronics Product",
    "misc.forsale":             "Billing/Marketplace",
}
categories = list(CATEGORY_LABELS.keys())

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),  # drop metadata that would leak the label trivially
    random_state=RANDOM_STATE,
)  # cached under sklearn's own data home (~/scikit_learn_data) on first run

docs = newsgroups.data
true_labels = np.array(newsgroups.target)
true_label_names = [CATEGORY_LABELS[categories[i]] for i in true_labels]

print(f"Documents fetched: {len(docs)}")
pd.Series(true_label_names).value_counts()

Documents fetched: 6880


Health/Medical Product    996
Billing/Marketplace       990
Motorcycle/Vehicle        990
Electronics Product       984
PC Hardware               982
Automotive                975
Mac Hardware              963
Name: count, dtype: int64

`remove=("headers", "footers", "quotes")` matters: newsgroup posts carry an `Xref`/`Newsgroups` header that would leak the true category directly into the text, and quoted replies duplicate another post's content. Stripping both forces every downstream feature to come from the actual complaint body, the same way a real ticket-routing system only has the customer's message to go on.

A side effect of removing quotes is a handful of posts become near-empty (a one-line reply that was pure quote) — those are dropped next.

In [3]:
mask = [len(d.strip()) > 20 for d in docs]
docs = [d for d, m in zip(docs, mask) if m]
true_labels = true_labels[mask]
true_label_names = [n for n, m in zip(true_label_names, mask) if m]
print(f"Documents after removing near-empty posts: {len(docs)}")

Documents after removing near-empty posts: 6630


`fetch_20newsgroups` caches the full raw corpus under scikit-learn's own data directory (`~/scikit_learn_data`) rather than this project's `data/` folder — the archive extracts to ~20,000 small files, which is exactly the kind of rapid-fire file creation that OneDrive's live sync does not handle gracefully inside a synced project folder. To keep this notebook reproducible without depending on that external cache (or a live download) on a future run, the filtered subset actually used is snapshotted to `data/` instead.

In [4]:
snapshot = pd.DataFrame({"text": docs, "category": true_label_names})
snapshot.to_parquet(f"{DATA_DIR}/complaint_topics_subset.parquet", index=False)
print(f"Snapshot saved: {len(snapshot)} documents -> {DATA_DIR}/complaint_topics_subset.parquet")

Snapshot saved: 6630 documents -> ./complaint_topics_subset.parquet


## 2. TF-IDF — Turning Words into Weighted Numbers

**TF-IDF** (Term Frequency - Inverse Document Frequency) weights each word by how informative it is: common *within this document* but rare *across all documents* scores high; words that appear in almost every document (or almost none) score low and are dropped entirely via `max_df`/`min_df`.

The stopword list below is scikit-learn's standard English list *plus* a short hand-picked extension, added for a concrete, evidence-based reason rather than a guess: an earlier pass through this pipeline using only the standard list produced two oversized, vague downstream clusters instead of seven clean ones, and reading their top terms showed why — informal filler words ("just", "like", "don't", "think", "know"...) common in casual newsgroup writing but carrying no topic information were dominating those clusters' centroids. Removing them is a direct, inspectable fix — the same "look at what the model is actually keying on, then fix that" loop applies whether the text is newsgroup posts or real support tickets.

In [5]:
FILLER_WORDS = [  # informal fillers found polluting downstream cluster centroids on an earlier pass
    "just", "like", "don", "ve", "does", "know", "think", "good",
    "did", "say", "said", "got", "going", "make", "want", "really",
]
stop_words = list(ENGLISH_STOP_WORDS) + FILLER_WORDS

vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    max_df=0.5,          # drop terms in >50% of documents — too common to be informative
    min_df=5,            # drop terms in fewer than 5 documents — too rare, likely noise/typos
    max_features=20000,
    sublinear_tf=True,   # log-scale raw term counts (1 + log(tf)) — keeps one very repeated word
                          # in a long post from dominating its vector; standard for uneven post lengths
)
X_tfidf = vectorizer.fit_transform(docs)
print(f"TF-IDF matrix: {X_tfidf.shape[0]} documents x {X_tfidf.shape[1]} terms")

TF-IDF matrix: 6630 documents x 9387 terms


## 3. Dimensionality Reduction — Truncated SVD (LSA)

The TF-IDF matrix is wide (thousands of terms) and sparse. Standard PCA needs to mean-center the data first, which destroys sparsity and becomes impractical at this width. **Truncated SVD** does the equivalent reduction directly on the sparse matrix — applied to a TF-IDF matrix, this is the classic **Latent Semantic Analysis (LSA)** technique: it collapses correlated terms (e.g. "engine", "horsepower", "mileage") into shared latent topics.

The output is then L2-normalized — a standard step before feeding LSA vectors into a distance-based algorithm like K-Means, since normalized vectors make Euclidean distance behave like cosine similarity, which is what actually matters for text.

In [6]:
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
normalizer = Normalizer(copy=False)
lsa = make_pipeline(svd, normalizer)

X_lsa = lsa.fit_transform(X_tfidf)
explained = svd.explained_variance_ratio_.sum()
print(f"Reduced {X_tfidf.shape[1]} terms -> {X_lsa.shape[1]} LSA dimensions, "
      f"explaining {explained:.1%} of the TF-IDF variance")

Reduced 9387 terms -> 100 LSA dimensions, explaining 13.3% of the TF-IDF variance


## 4. Save the Feature Matrix for Reuse

Saved as plain Parquet (`X_lsa` + labels) rather than a pickled sklearn object — portable, no scikit-learn version dependency, and it's all a clustering notebook actually needs to fit a model. The fitted `vectorizer`/`svd` objects themselves aren't persisted: nothing downstream currently maps cluster centroids back to raw terms, so there's no consumer for them — they'd just be dead weight sitting in `data/`. If that interpretation step comes back later, they're one `TfidfVectorizer`/`TruncatedSVD` re-fit away (a few seconds), since this notebook is the reproducible record of exactly how.

In [7]:
features_df = pd.DataFrame(X_lsa, columns=[f"feature_{i}" for i in range(X_lsa.shape[1])])
features_df["true_label"] = true_labels
features_df["true_label_name"] = true_label_names
features_df.to_parquet(f"{DATA_DIR}/complaint_features.parquet", index=False)
print(f"Feature matrix saved -> {DATA_DIR}/complaint_features.parquet  {features_df.shape}")

Feature matrix saved -> ./complaint_features.parquet  (6630, 102)
